# Importing Modules and Loading Data

In [ ]:
import json

USERS_FILE = "users.json"
TRIPS_FILE = "travel_plans.json"


def load_data(filename, default_data):
    """Load existing records from a JSON file."""

    file = None

    try:
        file = open(filename, "r")
        data = json.load(file)
        return data

    except FileNotFoundError:
        print(filename, "does not exist. Starting with empty data.")
        return default_data

    except json.JSONDecodeError:
        print("The file contains invalid JSON data.")
        return default_data

    except PermissionError:
        print("Permission denied while reading", filename)
        return default_data

    except OSError:
        print("An error occurred while reading the file.")
        return default_data

    finally:
        if file is not None:
            file.close()


# Load existing records when the program starts
users = load_data(USERS_FILE, {})
trips = load_data(TRIPS_FILE, [])

# Check that the loaded data has the correct structure
if not isinstance(users, dict):
    users = {}

if not isinstance(trips, list):
    trips = []


# Saving Data

In [ ]:
def save_data(filename, data):
    """Save updated records in a JSON file."""

    file = None

    try:
        file = open(filename, "w")
        json.dump(data, file, indent=4)
        return True

    except PermissionError:
        print("Permission denied while saving the file.")
        return False

    except OSError:
        print("An error occurred while saving data.")
        return False

    finally:
        if file is not None:
            file.close()


# User Registration

In [ ]:
def register_user(users, username, password):
    """Register a new user in the portal."""

    username = username.strip()
    password = password.strip()

    if username == "" or password == "":
        return False, "Username and password cannot be empty."

    if username in users:
        return False, "Username already exists."

    users[username] = password

    if save_data(USERS_FILE, users):
        return True, "Registration successful!"

    del users[username]
    return False, "Registration could not be completed."


# User Login

In [ ]:
def user_login(users, username, password):
    """Verify the user's login credentials."""

    if username in users and users[username] == password:
        return True

    return False

# Adding a Travel Plan

In [ ]:
def add_record(trips, username, destination, country,
               days, budget, trip_type, notes):
    """Add a new travel plan for a registered user."""

    destination = destination.strip()
    country = country.strip()
    trip_type = trip_type.strip()

    if destination == "" or country == "" or trip_type == "":
        return False, "Destination, country and trip type are required."

    try:
        days = int(days)
        budget = float(budget)

        if days <= 0 or budget <= 0:
            return False, "Days and budget must be positive."

    except ValueError:
        return False, "Please enter valid numerical values."

    travel_plan = {
        "username": username,
        "destination": destination,
        "country": country,
        "days": days,
        "budget": budget,
        "trip_type": trip_type,
        "notes": notes
    }

    trips.append(travel_plan)

    if save_data(TRIPS_FILE, trips):
        return True, "Travel plan added successfully!"

    trips.pop()
    return False, "Travel plan could not be saved."


# Viewing Travel Plans

In [ ]:
def view_records(trips, username):
    """Return all travel plans belonging to a user."""

    user_trips = []

    for trip in trips:

        if trip["username"] == username:
            user_trips.append(trip)

    return user_trips

# Searching for a Travel Plan

In [ ]:
def search_record(trips, username, destination):
    """Search for a destination in the user's travel plans."""

    results = []

    for trip in trips:

        if (trip["username"] == username and
            trip["destination"].lower() == destination.strip().lower()):

            results.append(trip)

    return results


# Generating a Travel Report

In [ ]:
def generate_report(trips, username):
    """Generate a summary of the user's travel plans."""

    user_trips = view_records(trips, username)

    total_trips = len(user_trips)
    total_budget = 0
    total_days = 0

    for trip in user_trips:

        total_budget += trip["budget"]
        total_days += trip["days"]

    if total_trips == 0:
        return None

    average_budget = total_budget / total_trips

    report = {
        "total_trips": total_trips,
        "total_budget": total_budget,
        "total_days": total_days,
        "average_budget": average_budget
    }

    return report


# Console Menu

In [ ]:
def run_console():
    """Run the TravelMate portal through a terminal menu."""

    while True:

        print("\n===== TRAVELMATE PORTAL =====")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        try:
            choice = int(input("Enter your choice: "))

        except ValueError:
            print("Please enter a valid number.")
            continue

        if choice == 1:

            username = input("Create username: ")
            password = input("Create password: ")

            success, message = register_user(users, username, password)
            print(message)

        elif choice == 2:

            username = input("Enter username: ")
            password = input("Enter password: ")

            if user_login(users, username, password):

                print("Login successful!")

                while True:

                    print("\n===== MAIN MENU =====")
                    print("1. Add Travel Plan")
                    print("2. View Travel Plans")
                    print("3. Search Travel Plan")
                    print("4. Generate Report")
                    print("5. Logout")

                    try:
                        option = int(input("Enter your choice: "))

                    except ValueError:
                        print("Enter a valid number.")
                        continue

                    if option == 1:

                        destination = input("Destination: ")
                        country = input("Country: ")
                        days = input("Number of days: ")
                        budget = input("Estimated budget: ")
                        trip_type = input("Trip type: ")
                        notes = input("Additional notes: ")

                        success, message = add_record(
                            trips, username, destination, country,
                            days, budget, trip_type, notes
                        )

                        print(message)

                    elif option == 2:

                        records = view_records(trips, username)

                        for trip in records:
                            print(trip)

                        if len(records) == 0:
                            print("No travel plans found.")

                    elif option == 3:

                        destination = input("Enter destination: ")
                        results = search_record(trips, username, destination)

                        for trip in results:
                            print(trip)

                        if len(results) == 0:
                            print("Destination not found.")

                    elif option == 4:

                        report = generate_report(trips, username)

                        if report is None:
                            print("No travel plans available.")

                        else:
                            print(report)

                    elif option == 5:
                        print("Logged out successfully!")
                        break

                    else:
                        print("Invalid option.")

            else:
                print("Incorrect username or password.")

        elif choice == 3:
            print("Thank you for using TravelMate!")
            break

        else:
            print("Invalid choice.")


# Run the console portal only when this file is executed directly
if __name__ == "__main__":
    run_console()
